# Filter pipeline walkthrough: ED → DMRG → MPS-to-circuit → dyadic filter → AA → QETU

Self-contained (numpy, scipy, quimb; qiskit optional for the KAK CNOT count). Small N, exact statevectors.

* **Model.** Open dimerised $J_1$-$J_2$ chain $H = J_1\sum_i (1+(-1)^i\delta)\,\mathbf S_i\cdot\mathbf S_{i+1} + J_2\sum_i \mathbf S_i\cdot\mathbf S_{i+2}$.
* **Filter variable.** $H_s = (H-E_0)/W$, $W = E_{\max}-E_0$, relative gap $\eta = \Delta/W$, $\tau = \pi/(1+\eta)$.
* **Dyadic filter.** $F(H_s) = \prod_{k<K}\cos(2^k\tau H_s)^r$, one ancilla-controlled pulse per factor.
* **Time evolution is exact here** (matrix exponentials, no Trotter), so the filter is isolated from Trotter error. Costs are given as calls to the controlled unit $e^{-i\tau H_s Z_a}$.

In [1]:
# ---- parameters (edit here) ----
N       = 8         # spins (even)
J1, J2  = 1.0, 0.2411
DELTA   = 0.15      # dimerisation: gap knob
GAMMA   = 0.2       # target overlap of the trial with the ground state: overlap knob
EPS     = 1e-2      # target infidelity 1 - F after the filter
TRIAL   = "rz"      # "rz": staggered Rz on the exact ground state (study); "mps1": staggered Rz on the 1-layer MPS circuit

import numpy as np, scipy.linalg as sla
from functools import reduce
from scipy.optimize import brentq, least_squares
np.set_printoptions(precision=6, suppress=True)
I2, X, Y, Z = np.eye(2), np.array([[0, 1], [1, 0]]), np.array([[0, -1j], [1j, 0]]), np.diag([1., -1.])

## 1. Hamiltonian and exact diagonalisation

In [2]:
def op(P, i):                                     # Pauli P on site i (site 0 = most significant bit)
    return reduce(np.kron, [P if k == i else I2 for k in range(N)])

def bonds(delta):
    b = [(i, i + 1, J1 * (1 + (-1) ** i * delta)) for i in range(N - 1)]
    return b + [(i, i + 2, J2) for i in range(N - 2)]

def hamiltonian(delta):                           # S.S = (XX + YY + ZZ) / 4
    return sum(J / 4 * (op(P, i) @ op(P, j)).real for i, j, J in bonds(delta) for P in (X, Y, Z))   # YY is real

def spectrum(delta):
    E, V = np.linalg.eigh(hamiltonian(delta))
    return E, V

Sz_tot = sum(op(Z, i) for i in range(N)).real / 2
print(f"N = {N}, J1 = {J1}, J2 = {J2}")
print(f"{'delta':>7} {'E0':>10} {'gap Δ':>8} {'W':>8} {'η = Δ/W':>9}")
for d in (-0.2, -0.1, 0.0, 0.1, DELTA, 0.3, 0.6):
    E, _ = spectrum(d)
    print(f"{d:7.3f} {E[0]:10.5f} {E[1] - E[0]:8.4f} {E[-1] - E[0]:8.4f} {(E[1] - E[0]) / (E[-1] - E[0]):9.5f}"
          + ("   <- chosen" if d == DELTA else ""))

H = hamiltonian(DELTA)
E, V = np.linalg.eigh(H)
E0, E1, Emax = E[0], E[1], E[-1]
g0 = V[:, 0]
Delta, W = E1 - E0, Emax - E0
eta, tau = Delta / W, np.pi / (1 + Delta / W)
deg = np.sum(abs(E - E1) < 1e-9)
print(f"\nchosen delta = {DELTA}:  E0 = {E0:.8f}, E1 = {E1:.8f} ({deg}-fold, triplet), Emax = {Emax:.6f}")
print(f"gap Δ = {Delta:.6f},  W = {W:.6f},  η = Δ/W = {eta:.6f},  τ = π/(1+η) = {tau:.6f}")
print(f"<S^z_tot> of ground state = {g0 @ Sz_tot @ g0:.1e}  (singlet, S^z = 0 sector)")

N = 8, J1 = 1.0, J2 = 0.2411
  delta         E0    gap Δ        W   η = Δ/W
 -0.200   -2.93313   0.0448   4.9948   0.00896
 -0.100   -2.96567   0.1730   5.0523   0.03424


  0.000   -3.13500   0.3859   5.2467   0.07355
  0.100   -3.36959   0.6146   5.5062   0.11162
  0.150   -3.49903   0.7256   5.6482   0.12846   <- chosen


  0.300   -3.91361   1.0378   6.1003   0.17013
  0.600   -4.80138   1.5641   7.0630   0.22145

chosen delta = 0.15:  E0 = -3.49903410, E1 = -2.77345583 (3-fold, triplet), Emax = 2.149150
gap Δ = 0.725578,  W = 5.648184,  η = Δ/W = 0.128462,  τ = π/(1+η) = 2.783959
<S^z_tot> of ground state = 2.6e-29  (singlet, S^z = 0 sector)


## 2. DMRG on the same Hamiltonian, spectral errors vs ED, and the 1-layer MPS-to-circuit trial

* DMRG gives $E_0$; $E_{\max}$ from DMRG on $-H$; $E_1$ from DMRG with a penalty $\lambda(S^z_{tot}-1)^2$ (lowest triplet).
* MPS-to-circuit: truncate the DMRG state to bond dimension 2 (S^z-conserving SVD), then each bond-2 tensor is one
  two-qubit gate in a staircase of $N-1$ gates on $|0\cdots0\rangle$. Its overlap with the ED ground state is $\gamma_0$.

In [3]:
import quimb.operator as qop, quimb.tensor as qtn

def builder(sign=1.0, penalty=None):
    Jd = {(i, j): sign * J for i, j, J in bonds(DELTA)}
    b = qop.heisenberg_from_edges(list(Jd), j=lambda i, j: Jd[(i, j)] if (i, j) in Jd else Jd[(j, i)])
    if penalty:                                   # lam (Sz - 1)^2 = lam [sum_{i!=j} sz_i sz_j - 2 Sz] + lam (N/4 + 1)
        for i in range(N):
            for j in range(N):
                if i != j:
                    b.add_term(penalty, ("sz", i), ("sz", j))
            b.add_term(-2 * penalty, ("sz", i))
    return b.build_mpo()

def dmrg(mpo):
    d = qtn.DMRG2(mpo, bond_dims=[16, 32, 64], cutoffs=1e-12)
    d.solve(tol=1e-10, verbosity=0, max_sweeps=40)
    return float(np.real(d.energy)), d.state

E0_d, psi_d = dmrg(builder())
Emax_d = -dmrg(builder(-1.0))[0]
lam = 2.0
E1_d = dmrg(builder(penalty=lam))[0] + lam * (N / 4 + 1)
g0_d = psi_d.to_dense().ravel(); g0_d /= np.linalg.norm(g0_d)
print(f"{'':6} {'ED':>14} {'DMRG':>14} {'error':>10}")
for name, a, b in (("E0", E0, E0_d), ("E1", E1, E1_d), ("Emax", Emax, Emax_d),
                   ("Δ", Delta, E1_d - E0_d), ("η", eta, (E1_d - E0_d) / (Emax_d - E0_d))):
    print(f"{name:6} {a:14.9f} {b:14.9f} {b - a:10.1e}")
print(f"ground state infidelity 1 - |<ED|DMRG>|^2 = {1 - abs(np.vdot(g0, g0_d)) ** 2:.1e}, max bond {psi_d.max_bond()}")

# ---- 1-layer MPS-to-circuit
def truncate_bond2(full):
    # right-canonical bond-2 MPS, S^z kept: SVD block by block in the S^z of the right part; ties -> smallest |S^z|
    B, psi, q_right = [None] * N, full.reshape(-1, 2), np.array([0.0])
    for i in range(N - 1, 0, -1):
        chi_r = len(q_right)
        cols_q = np.array([0.5 - s + q for s in (0, 1) for q in q_right])
        sv, us, vs, qs = [], [], [], []
        for q in np.unique(cols_q):
            idx = np.flatnonzero(np.isclose(cols_q, q))
            U, S, Vh = np.linalg.svd(psi[:, idx], full_matrices=False)
            for k in range(len(S)):
                v = np.zeros(2 * chi_r, complex); v[idx] = Vh[k]
                sv.append(S[k]); us.append(U[:, k]); vs.append(v); qs.append(q)
        sv, qs = np.array(sv), np.array(qs)
        keep = sorted(range(len(sv)), key=lambda k: (-np.round(sv[k] / sv.max() / 1e-8), abs(qs[k])))[:2]
        B[i] = np.array([vs[k] for k in keep]).reshape(len(keep), 2, chi_r)
        left = np.stack([us[k] * sv[k] for k in keep], axis=1)
        q_right = qs[keep]
        psi = left.reshape(-1, 2 * len(keep))
    B[0] = psi.reshape(1, 2, -1)
    return B

def staircase(B):
    # gate G_i on (i, i+1): |a>_i|0>_{i+1} -> sum_{s,b} B_i[a,s,b] |s>_i|b>_{i+1}; isometry completed to a unitary
    pad = lambda T: np.pad(T, ((0, 2 - T.shape[0]), (0, 0), (0, 2 - T.shape[2])))
    Bs = [pad(T) for T in B]; Bs[0] = Bs[0] / np.linalg.norm(Bs[0])
    gates = []
    for i in range(N - 1):
        T = Bs[i] if i < N - 2 else np.einsum("asb,tb->ast", Bs[i], Bs[N - 1][:, :, 0].T)
        iso = T.reshape(2, 4).T
        iso = iso[:, :1] if i == 0 else iso[:, np.linalg.norm(iso, axis=0) > 1e-12]
        Q, _ = np.linalg.qr(np.hstack([iso, np.random.default_rng(0).normal(size=(4, 4 - iso.shape[1]))]))
        Q[:, :iso.shape[1]] = iso
        G = np.zeros((4, 4), complex); ins = [0, 2][:iso.shape[1]]
        G[:, ins] = Q[:, :len(ins)]; G[:, [x for x in range(4) if x not in ins]] = Q[:, len(ins):]
        gates.append(G)
    return gates

def run_staircase(gates):
    v = np.zeros(2 ** N, complex); v[0] = 1
    for i, G in enumerate(gates):
        v = np.einsum("xy,ayb->axb", G, v.reshape(2 ** i, 4, -1)).ravel()
    return v

gates = staircase(truncate_bond2(g0_d))
psi_mps1 = run_staircase(gates)
gamma0 = abs(np.vdot(g0, psi_mps1)) ** 2
try:
    from qiskit.circuit.library import CXGate
    from qiskit.synthesis import TwoQubitBasisDecomposer
    kak = TwoQubitBasisDecomposer(CXGate())
    cnots = sum(int(kak.num_basis_gates(G)) for G in gates)
except ImportError:
    cnots = 3 * (N - 1)
print(f"\n1-layer MPS-to-circuit: {N - 1} two-qubit gates, {cnots} CNOTs (KAK)")
print(f"  <S^z_tot> = {np.vdot(psi_mps1, Sz_tot @ psi_mps1).real:.1e},  overlap with ED ground state γ0 = {gamma0:.6f}")

                   ED           DMRG      error
E0       -3.499034099   -3.499034099   -4.4e-16
E1       -2.773455831   -2.773455831    5.6e-14
Emax      2.149150000    2.149150000    3.1e-15
Δ         0.725578269    0.725578269    5.6e-14
η         0.128462220    0.128462220    9.9e-15
ground state infidelity 1 - |<ED|DMRG>|^2 = -8.9e-16, max bond 16



1-layer MPS-to-circuit: 7 two-qubit gates, 16 CNOTs (KAK)
  <S^z_tot> = -4.3e-24,  overlap with ED ground state γ0 = 0.977843


## 3. Trial state and the dyadic filter (repeat until success, N + 1 qubits)

Trial: $e^{-i\epsilon\sum_i(-1)^iZ_i}$ on the base state, with $\epsilon$ solved so $|\langle E_0|\text{trial}\rangle|^2 = \gamma$.
Each pulse: H on the ancilla, $e^{-i\,2^k\tau H_s Z_a}$, H, measure the ancilla; outcome 0 applies $\cos(2^k\tau H_s)$.
The cheapest $(K, r)$ reaching $1-F\le\epsilon$ is picked from the exact filter function, then the ancilla circuit is run.

In [4]:
Hs_E = (E - E0) / W                                     # eigenvalues of H_s
def evolve(t, v):                                       # e^{-i t H_s} v, exact (v may carry leading batch axes)
    return ((v @ V.conj()) * np.exp(-1j * t * Hs_E)) @ V.T

stag = np.array([sum((-1) ** i * (1 - 2 * ((b >> (N - 1 - i)) & 1)) for i in range(N)) for b in range(2 ** N)])
base = g0 if TRIAL == "rz" else psi_mps1
trial_of = lambda e: np.exp(-1j * e * stag) * base
ov = lambda e: abs(np.vdot(g0, trial_of(e))) ** 2
es = np.linspace(0, 1.5, 301); first_min = es[np.argmax(np.diff([ov(e) for e in es]) > 0)]
eps_knob = brentq(lambda e: ov(e) - GAMMA, 0, first_min)
trial = trial_of(eps_knob)
print(f"trial: {TRIAL}, Rz angle ε_knob = {eps_knob:.6f}, overlap γ = {ov(eps_knob):.6f}")

def filter_fn(x, K, r):                                 # F on eigenvalues of H_s
    return np.prod([np.cos(2 ** k * tau * x) for k in range(K)], axis=0) ** r

c = V.conj().T @ trial                                  # trial in the eigenbasis
def p_and_infid(K, r):
    f = filter_fn(Hs_E, K, r); p = np.sum(abs(c * f) ** 2)
    return p, 1 - abs(c[0] * f[0]) ** 2 / p

best = min(((r * (2 ** K - 1) / p_and_infid(K, r)[0], K, r) for K in range(1, 14) for r in range(1, 10)
            if p_and_infid(K, r)[1] <= EPS), default=None)
assert best, "no (K, r) reaches EPS"
_, K, r = best
print(f"cheapest dyadic filter for 1-F <= {EPS}: K = {K}, r = {r}  ->  {K * r} pulses, "
      f"d = r(2^K-1) = {r * (2 ** K - 1)} calls of e^(-iτ H_s Z_a)")

# ---- run it as a circuit: one ancilla, measured and reset after every pulse
psi, p_tot = trial.copy(), 1.0
print(f"\n{'pulse':>5} {'k':>2} {'P(anc = 0)':>11} {'1 - F so far':>13}")
for j, k in enumerate([k for _ in range(r) for k in range(K)]):
    a0, a1 = psi / np.sqrt(2), psi / np.sqrt(2)                       # H on |0>_a
    a0, a1 = evolve(2 ** k * tau, a0), evolve(-(2 ** k) * tau, a1)   # e^{-i 2^k τ H_s Z_a}
    out0 = (a0 + a1) / np.sqrt(2)                                     # H, keep ancilla = 0
    pk = np.vdot(out0, out0).real; p_tot *= pk; psi = out0 / np.sqrt(pk)
    print(f"{j + 1:5d} {k:2d} {pk:11.6f} {1 - abs(np.vdot(g0, psi)) ** 2:13.3e}")
p_rus, inf_rus = p_and_infid(K, r)
print(f"\nRUS: success p = {p_tot:.6f} (formula {p_rus:.6f}),  1 - F = {1 - abs(np.vdot(g0, psi)) ** 2:.3e}")
print(f"cost per success = d/p = {r * (2 ** K - 1) / p_tot:.1f} controlled-τ calls,  qubits N + 1 = {N + 1}")

trial: rz, Rz angle ε_knob = 0.277011, overlap γ = 0.200000
cheapest dyadic filter for 1-F <= 0.01: K = 3, r = 2  ->  6 pulses, d = r(2^K-1) = 14 calls of e^(-iτ H_s Z_a)

pulse  k  P(anc = 0)  1 - F so far
    1  0    0.628542     6.818e-01
    2  1    0.611108     4.793e-01
    3  2    0.550405     5.399e-02
    4  0    0.975525     3.026e-02
    5  1    0.979760     1.022e-02
    6  2    0.991130     1.365e-03

RUS: success p = 0.200273 (formula 0.200273),  1 - F = 1.365e-03
cost per success = d/p = 69.9 controlled-τ calls,  qubits N + 1 = 9


## 4. Amplitude amplification on the dyadic filter (N + K·r qubits)

No mid-circuit measurement, so each pulse gets its own ancilla. $A$ = all pulses on $|\text{trial}\rangle|0\cdots0\rangle$;
one round $Q = -A\,R_0\,A^\dagger R_{good}$ with $R_{good}$ = sign flip of "all ancillas 0" and $A R_0 A^\dagger$ = reflection about the
start state. $k = \text{round}(\pi/4\theta - 1/2)$, $\sin\theta = \sqrt p$.

In [5]:
m = K * r
ks = [k for _ in range(r) for k in range(K)]
def apply_pulses(st, dagger=False):                      # st: shape (2,)*m + (2^N,)
    for j in (reversed(range(m)) if dagger else range(m)):
        st = np.moveaxis(st, j, 0)
        a0, a1 = (st[0] + st[1]) / np.sqrt(2), (st[0] - st[1]) / np.sqrt(2)     # H
        t = 2 ** ks[j] * tau * (-1 if dagger else 1)
        a0, a1 = evolve(t, a0), evolve(-t, a1)
        st = np.moveaxis(np.stack([(a0 + a1) / np.sqrt(2), (a0 - a1) / np.sqrt(2)]), 0, j)
    return st

start = np.zeros((2,) * m + (2 ** N,), complex); start[(0,) * m] = trial
st = apply_pulses(start)
theta = np.arcsin(np.sqrt(p_rus)); k_aa = int(round(np.pi / (4 * theta) - 0.5))
print(f"qubits N + K·r = {N + m},  p = {p_rus:.6f},  θ = {theta:.4f},  rounds k = {k_aa}")
print(f"{'round':>5} {'P(good) sim':>12} {'sin²((2j+1)θ)':>14} {'1 - F':>10}")
for j in range(k_aa + 2):
    good = st[(0,) * m]
    pg = np.vdot(good, good).real
    print(f"{j:5d} {pg:12.6f} {np.sin((2 * j + 1) * theta) ** 2:14.6f} {1 - abs(np.vdot(g0, good)) ** 2 / pg:10.3e}"
          + ("   <- k" if j == k_aa else ""))
    st[(0,) * m] *= -1                                                   # R_good
    st = apply_pulses(st, dagger=True)                                   # A^dagger
    st = st - 2 * np.vdot(start.ravel(), st.ravel()) * start             # R_0 (reflection about the start state)
    st = -apply_pulses(st)                                               # -A
d = r * (2 ** K - 1); pk = np.sin((2 * k_aa + 1) * theta) ** 2
print(f"\ncost per success, controlled-τ calls:  RUS d/p = {d / p_rus:.1f},   AA (2k+1)d / P_k = {(2 * k_aa + 1) * d / pk:.1f}"
      "   (reflections not included)")

qubits N + K·r = 14,  p = 0.200273,  θ = 0.4640,  rounds k = 1
round  P(good) sim  sin²((2j+1)θ)      1 - F
    0     0.200273       0.200273  1.365e-03
    1     0.968360       0.968360  1.365e-03   <- k
    2     0.536217       0.536217  1.365e-03

cost per success, controlled-τ calls:  RUS d/p = 69.9,   AA (2k+1)d / P_k = 43.4   (reflections not included)


## 5. QETU: the same filter polynomial on N + 1 qubits, no mid-circuit measurement

$U_\Phi = e^{i\phi_0X}\,\prod_{j=1}^{d}\big(e^{-i\tau H_sZ_a}e^{i\phi_jX}\big)$. On an eigenstate $|\lambda\rangle$ the ancilla sees
$e^{-i\theta Z}$, $\theta=\tau\lambda$, so $\langle0|U_\Phi|0\rangle = P(\cos\tau\lambda)$ (QSP). Since $\cos(2^k\theta)=T_{2^k}(\cos\theta)$ the dyadic filter is the
polynomial $F(x)=\prod_k T_{2^k}(x)^r$ of degree $d=r(2^K-1)$; the phases $\phi$ are solved so that $P = F$. AA then needs
only a single-qubit $R_{good}$ (Z on the ancilla).

In [6]:
def rx(p): return np.cos(p) * I2 + 1j * np.sin(p) * X
def sig(th):                                            # signal e^{-iθZ} for an array of θ: shape (n, 2, 2)
    S = np.zeros((len(th), 2, 2), complex); S[:, 0, 0], S[:, 1, 1] = np.exp(-1j * th), np.exp(1j * th)
    return S
def qsp(phi, th, grad=False):
    # <0|U_Phi|0> on an array of θ, U_Phi = R0 S R1 S ... S Rd; with grad also d/dφ_j via prefix/suffix products
    S, R = sig(np.atleast_1d(th)), [rx(p) for p in phi]
    pre = [np.broadcast_to(I2, S.shape)]
    for j in range(len(phi) - 1):
        pre.append(pre[-1] @ R[j] @ S)
    U = pre[-1] @ R[-1]
    if not grad:
        return U[:, 0, 0]
    suf = [np.broadcast_to(I2, S.shape)]
    for j in range(len(phi) - 1, 0, -1):
        suf.append(S @ R[j] @ suf[-1])
    suf = suf[::-1]
    G = np.stack([(pre[j] @ (1j * X @ R[j]) @ suf[j])[:, 0, 0] for j in range(len(phi))], axis=1)
    return U[:, 0, 0], G

d = r * (2 ** K - 1)
nodes = np.pi * (np.arange(2 * d + 2) + 0.5) / (2 * d + 2)          # θ on Chebyshev nodes
Fx = lambda th: np.prod([np.cos(2 ** k * th) for k in range(K)], axis=0) ** r
def resid(phi):
    v = qsp(phi, nodes) - Fx(nodes)
    return np.concatenate([v.real, v.imag])
def jac(phi):
    G = qsp(phi, nodes, grad=True)[1]
    return np.vstack([G.real, G.imag])
for seed in range(50):                                  # least squares from random starts until P = F
    sol = least_squares(resid, np.random.default_rng(seed).normal(scale=0.5, size=d + 1), jac=jac, xtol=1e-15,
                        ftol=1e-15, gtol=1e-15, max_nfev=5000)
    if sol.cost < 1e-24:
        break
phi = sol.x
chk = np.linspace(0, np.pi, 997)
print(f"degree d = {d}, phases solved: max |P(cos θ) - F(cos θ)| on a fine grid = "
      f"{np.max(abs(qsp(phi, chk) - Fx(chk))):.1e}  (random starts: {seed + 1})")

# ---- the actual N + 1 qubit circuit: state (2, 2^N) = (ancilla, system)
def rx_anc(st, p):
    return np.stack([np.cos(p) * st[0] + 1j * np.sin(p) * st[1], 1j * np.sin(p) * st[0] + np.cos(p) * st[1]])
def ctrl(st, sgn=1):                                    # e^{-i sgn τ H_s Z_a}
    return np.stack([evolve(sgn * tau, st[0]), evolve(-sgn * tau, st[1])])
def U_phi(st):
    st = rx_anc(st, phi[0])
    for p in phi[1:]:
        st = rx_anc(ctrl(st), p)
    return st
def U_phi_dag(st):
    for p in phi[:0:-1]:
        st = ctrl(rx_anc(st, -p), -1)
    return rx_anc(st, -phi[0])

st0 = np.stack([trial, np.zeros_like(trial)])
st = U_phi(st0)
p_q = np.vdot(st[0], st[0]).real
print(f"\nQETU, RUS: qubits N + 1 = {N + 1},  p = {p_q:.6f} (dyadic {p_rus:.6f}),  "
      f"1 - F = {1 - abs(np.vdot(g0, st[0])) ** 2 / p_q:.3e} (dyadic {inf_rus:.3e}),  calls = {d}")

print(f"\nQETU + AA, k = {k_aa}:")
print(f"{'round':>5} {'P(good) sim':>12} {'sin²((2j+1)θ)':>14} {'1 - F':>10}")
for j in range(k_aa + 2):
    pg = np.vdot(st[0], st[0]).real
    print(f"{j:5d} {pg:12.6f} {np.sin((2 * j + 1) * theta) ** 2:14.6f} {1 - abs(np.vdot(g0, st[0])) ** 2 / pg:10.3e}"
          + ("   <- k" if j == k_aa else ""))
    st = st * np.array([-1, 1])[:, None]                                 # R_good: Z on the ancilla
    st = U_phi_dag(st)
    st = st - 2 * np.vdot(st0.ravel(), st.ravel()) * st0                 # reflection about the start state
    st = -U_phi(st)
print(f"\nsame success and fidelity as the dyadic AA, on {N + 1} qubits instead of {N + m}")

degree d = 14, phases solved: max |P(cos θ) - F(cos θ)| on a fine grid = 1.1e-15  (random starts: 1)



QETU, RUS: qubits N + 1 = 9,  p = 0.200273 (dyadic 0.200273),  1 - F = 1.365e-03 (dyadic 1.365e-03),  calls = 14

QETU + AA, k = 1:
round  P(good) sim  sin²((2j+1)θ)      1 - F
    0     0.200273       0.200273  1.365e-03
    1     0.968360       0.968360  1.365e-03   <- k
    2     0.536217       0.536217  1.365e-03

same success and fidelity as the dyadic AA, on 9 qubits instead of 14
